In [1]:
import pandas as pd
import numpy as np
# 행(row) 다 보기
pd.set_option('display.max_rows', None)

# 열(column) 다 보기
pd.set_option('display.max_columns', None)

In [80]:
def recent_and_past(s):
    if s is None or len(s) == 0:
        return 0.0, 0.0, 0.0
    try:
        return float(s.iloc[0]), float(s.iloc[1]), float(s.iloc[-1])
    except Exception:
        try:
            r = s.iloc[0]
            r1 = s.iloc[1]
            p = s.iloc[-1]
            r = float(r) if not pd.isna(r) else 0.0
            r1 = float(r1) if not pd.isna(r1) else 0.0
            p = float(p) if not pd.isna(p) else 0.0
            return r, r1, p
        except Exception:
            return 0.0, 0.0, 0.0

def is_exist_col(dict_, doc, col):
    if col in dict_[doc].keys():
        return True
    else:
        return False

In [2]:
dict_us_stocks = pd.read_pickle('yf_chunk_all.pkl')

df = pd.read_pickle('bs_fin.pickle')
df['per'] = df['per'].replace([np.inf, -np.inf], 0)
df['payout(%)'] = np.round(df['payout(%)'] * 100, 2)
df.head()

,Symbol,sector,industry,market_cap,price,eps,pbs,per,pbr,psr,roe,net_income,ev/ebitda,freecashflow,is_share_reduced,is_share_same,is_eps_inc,is_revenue_inc,is_operating_income_inc,is_net_income_inc,is_debt_inc,is_debt_long_inc,is_debt_short_inc,is_debt_payables_inc,is_debt_deferredTax,total_revenue,current(%),quick(%),op(%),net(%),dividend_rate,dividend_yield,payout(%),debt(%),debt_long(%),debt_short(%),debt_payables(%),debt_tax(%),is_ok_payables,is_ok_tax
0,CRESY,Industrials,Conglomerates,825891008,11.83,1.81,1.207939,6.535912,9.793539,0.000858,0.19706,3.684400e+10,13.853,-73069625344,False,False,True,True,True,False,True,True,True,True,False,3.185290e+11,133.07,101.43,19.12,11.57,0.63,5.32,23.40,305.27,34.19,15.09,12.52,0.0,True,True
1,SCHL,Communication Services,Publishing,728269824,28.97,-0.69,34.926000,0.000000,0.829468,0.451248,-0.01144,-7.110000e+07,13.513,69587504,False,False,False,False,False,False,True,True,True,False,False,2.256000e+08,115.65,64.62,-40.51,-31.52,0.80,2.76,135.59,122.62,38.11,0.58,16.33,0.0,True,True
2,GYRE,Healthcare,Biotechnology,740801984,7.69,0.04,1.121000,192.250000,6.859946,6.906279,0.10017,3.610000e+06,47.048,2900250,False,False,True,True,True,True,False,False,False,True,False,3.056400e+07,619.47,563.83,22.66,11.81,0.00,0.00,0.00,21.56,2.45,0.00,2.53,0.0,True,True
3,LINC,Consumer Defensive,Education & Training Services,696356032,22.02,0.45,5.877000,48.933334,3.746810,1.407513,0.07943,3.799000e+06,21.105,-36984376,False,False,False,True,True,False,True,True,False,True,False,1.413890e+08,80.66,77.43,4.45,2.69,0.00,0.00,0.00,151.24,65.89,0.00,12.48,0.0,True,True
4,EMBC,Healthcare,Medical Instruments & Supplies,690451520,11.80,1.62,-11.122000,7.283951,-1.060960,0.639070,0.00000,2.640000e+07,5.372,183150000,False,False,True,False,True,True,True,False,False,False,False,2.640000e+08,241.45,173.15,24.36,10.00,0.60,5.08,37.04,-267.52,81.36,0.55,4.26,0.0,True,True


In [130]:
all_metrics = []
list_bs_fin_tickers = df['Symbol'].tolist()
for ticker in list_bs_fin_tickers:
    tkr = dict_us_stocks[ticker]

    #################################################################
    #  revenue growth(매출성장률) ==> 매출 고성장
    #################################################################
    rev_yoy_growth = None
    rev_prev_growth = None
    rev_recent = None
    rev_prev = None
    rev_past = None
    if is_exist_col(tkr, 'financials', 'Total Revenue'):

        rev_recent, rev_prev, rev_past = recent_and_past(tkr['financials']['Total Revenue'][:5])
        if str(rev_recent).lower() == 'nan': continue
        if str(rev_past).lower() == 'nan': continue
        if rev_recent < 1: continue
        if rev_past < 1: continue
        rev_yoy_growth = np.round((rev_recent - rev_past) / rev_past * 100, 2)
       # rev_prev_growth = np.round((rev_recent - rev_prev) / rev_prev * 100, 2)

    #################################################################
    #  fcf & fcf margin ==> 현금흐름 플러스
    #################################################################
    fcf_recent_margin = None
    fcf_prev_margin = None
    fcf_path_margin = None
    capex_recent = None
    capex_prev = None
    capex_past = None
    if is_exist_col(tkr, 'cashflow', 'Operating Cash Flow') and is_exist_col(tkr, 'cashflow', 'Capital Expenditure'):
        ocf_recent, ocf_prev, ocf_past = recent_and_past(tkr['cashflow']['Operating Cash Flow'][:5])
        capex_recent, capex_prev, capex_past = recent_and_past(tkr['cashflow']['Capital Expenditure'][:5])

        fcf_recent = ocf_recent - capex_recent
        fcf_prev = ocf_prev - capex_prev
        fcf_past = ocf_past - capex_past

        fcf_recent_margin = np.round(fcf_recent / rev_recent * 100, 2)
        fcf_prev_margin = np.round(fcf_prev / rev_recent * 100, 2)
        fcf_past_margin = np.round(fcf_past / rev_recent * 100, 2)

        fcf_growth = np.round((fcf_recent_margin - fcf_past_margin) / fcf_past_margin * 100, 2)

    #################################################################
    #  grosss profit margin (매출 - 매출원가) ==> 마진 우수
    #################################################################
    gpf_recent_margin = None
    gpf_prev_margin = None
    gpf_past_margin = None
    if is_exist_col(tkr, 'incomestmt', 'Gross Profit'):
        gpf_recent, gpf_prev, gpf_past = recent_and_past(tkr['incomestmt']['Gross Profit'][:5])

        gpf_recent_margin = np.round(gpf_recent / rev_recent * 100, 2)
        #gpf_prev_margin = np.round(gpf_prev / rev_prev * 100, 2)
        gpf_past_margin = np.round(gpf_past / rev_past * 100, 2)
        gpf_growth = np.round((gpf_recent_margin - gpf_past_margin) / gpf_past_margin * 100, 2)

    #################################################################
    #  capex / revenue (성장의지) ==> 투자 지속
    #################################################################
    capex_recent_growth = None
    capex_prev_growth = None
    capex_past_growth = None
    if capex_recent is not None:
        capex_recent_growth = np.round(np.abs(capex_recent) / rev_recent * 100, 2)
        capex_prev_growth = np.round(np.abs(capex_prev) / rev_prev * 100, 2)
        capex_past_growth = np.round(np.abs(capex_past) / rev_past * 100, 2)
        capex_growth = np.round((capex_recent_growth - capex_past_growth) / capex_past_growth * 100, 2)

    #################################################################
    #  net debit / revenue (안정성) ==> 부채 안정
    #################################################################
    safety_recent = None
    safety_prev = None
    safety_past = None
    if is_exist_col(tkr, 'balance_sheet', 'Total Debt') \
        and is_exist_col(tkr, 'balance_sheet', 'Cash Cash Equivalents And Short Term Investments') \
        and is_exist_col(tkr, 'financials', 'EBITDA'):

        debt_recent, debt_prev, debt_past = recent_and_past(tkr['balance_sheet']['Total Debt'][:5])
        cash_recent, cash_prev, cash_past = recent_and_past(tkr['balance_sheet']['Cash Cash Equivalents And Short Term Investments'][:5])
        ebitda_recent, ebitda_prev, ebitda_past = recent_and_past(tkr['financials']['EBITDA'][:5])

        safety_recent = np.round((debt_recent - cash_recent) / ebitda_recent * 100, 2)
        safety_prev = np.round((debt_prev - cash_prev) / ebitda_prev * 100, 2)
        safety_past = np.round((debt_past - cash_past)  / ebitda_past * 100, 2)

    #################################################################
    #  주식 수 ==> 희석 통제
    #################################################################
    share_growth = None
    if is_exist_col(tkr, 'balance_sheet', 'Share Issued'):
        share_recent, share_prev, share_past = recent_and_past(tkr['balance_sheet']['Share Issued'][:5])
        share_growth = np.round((share_recent - share_past) / share_past * 100, 2)



    metrics = {
        'Symbol' : ticker,
        'revenue' : rev_recent,
        'revenue_growth' : rev_yoy_growth,
        'gpf_growth' : gpf_growth,
        'fcf_growth' : fcf_growth,
        'capex_growth' : capex_growth,
        'fcf_recent_margin' : fcf_recent_margin,
       # 'fcf_prev_margin' : fcf_prev_margin,
        'fcf_past_margin' : fcf_past_margin,
        'gpf_recent_margin' : gpf_recent_margin,
        #'gpf_prev_margin' : gpf_prev_margin,
        'gpf_past_margin' : gpf_past_margin,
        'capex_recent_growth' : capex_recent_growth,
      #  'capex_prev_growth' : capex_prev_growth,
        'capex_past_growth' : capex_past_growth,
        'safety_recent' : safety_recent,
       # 'safety_prev' : safety_prev,
        'safety_past' : safety_past,
        'share_growth' : share_growth
    }
    all_metrics.append(metrics)


/var/folders/jk/1j1mgc7x11122bdp64mnjcf40000gp/T/ipykernel_12509/2194149085.py:45: RuntimeWarning: invalid value encountered in scalar divide
  fcf_growth = np.round((fcf_recent_margin - fcf_past_margin) / fcf_past_margin * 100, 2)
/var/folders/jk/1j1mgc7x11122bdp64mnjcf40000gp/T/ipykernel_12509/2194149085.py:71: RuntimeWarning: invalid value encountered in scalar divide
  capex_growth = np.round((capex_recent_growth - capex_past_growth) / capex_past_growth * 100, 2)


In [131]:
df_metrics = pd.DataFrame(all_metrics)
df_metrics = df[['Symbol', 'sector', 'industry', 'market_cap']].merge(df_metrics, on='Symbol', how='left')
df_metrics.head()

,Symbol,sector,industry,market_cap,revenue,revenue_growth,gpf_growth,fcf_growth,capex_growth,fcf_recent_margin,fcf_past_margin,gpf_recent_margin,gpf_past_margin,capex_recent_growth,capex_past_growth,safety_recent,safety_past,share_growth
0,CRESY,Industrials,Conglomerates,825891008,3.185290e+11,22.25,9.17,386.56,-59.28,51.77,10.64,35.84,32.83,4.06,9.97,451.51,535.84,5.95
1,SCHL,Communication Services,Publishing,728269824,2.256000e+08,-4.89,-1.42,227.81,-47.45,-31.83,-9.71,45.26,45.91,4.43,8.43,-484.58,-374.36,0.00
2,GYRE,Healthcare,Biotechnology,740801984,3.056400e+07,19.92,-1.63,121.34,-18.32,17.53,7.92,94.67,96.24,2.23,2.73,-777.35,-498.01,5.97
3,LINC,Consumer Defensive,Education & Training Services,696356032,1.413890e+08,23.58,2.57,83.18,-8.74,32.35,17.66,59.49,58.00,15.45,16.93,1608.82,1285.92,0.46
4,EMBC,Healthcare,Medical Instruments & Supplies,690451520,2.640000e+08,-7.72,-1.17,NaN,NaN,34.58,NaN,60.04,60.75,2.77,NaN,1994.70,4840.79,1.37


### 거시면적 측면

In [132]:
df_sector = (
    df_metrics.groupby(['sector', 'industry'])
    .agg({'revenue_growth':'mean',
          'fcf_recent_margin':'mean',
          'gpf_recent_margin':'mean',
          'capex_recent_growth':'mean',
          'safety_recent':'mean',
          'market_cap':'mean'})
    .reset_index()
)

df_sector.sort_values(by=['revenue_growth', 'gpf_recent_margin', 'fcf_recent_margin', 'capex_recent_growth'], ascending=False).head(10)

,sector,industry,revenue_growth,fcf_recent_margin,gpf_recent_margin,capex_recent_growth,safety_recent,market_cap
86,Industrials,Industrial Distribution,1732.129167,11.535833,30.806667,1.580000,635.619167,1.753858e+10
77,Healthcare,Pharmaceutical Retailers,394.690000,-0.730000,62.765000,1.470000,-2217.715000,4.371214e+07
119,Technology,Software - Application,153.742857,19.591667,54.161429,3.995000,144.460000,9.432522e+09
67,Healthcare,Biotechnology,127.875714,30.244000,74.920000,4.088000,-370.840000,1.298133e+10
11,Basic Materials,Silver,95.060000,59.240000,34.560000,20.000000,-268.620000,7.816275e+09
81,Industrials,Business Equipment & Supplies,48.603333,19.456667,24.643333,4.353333,-288.370000,3.952783e+08
63,Financial Services,Asset Management,46.023333,22.800000,6.690000,11.960000,2320.060000,3.507073e+10
6,Basic Materials,Gold,43.239412,99.137059,54.540588,46.270000,28.711875,3.164259e+10
33,Consumer Cyclical,Luxury Goods,42.808000,-0.870000,53.826000,1.084000,-155.114000,6.208326e+09
9,Basic Materials,Other Precious Metals & Mining,41.332500,101.360000,43.352500,57.195000,103401.570000,6.994674e+09


### 미시적 측면

In [134]:
lvl_rev_growth = df_metrics['revenue_growth'].describe()['75%']
lvl_gpf_growth = df_metrics['gpf_growth'].describe()['75%']
lvl_fcf_growth = df_metrics['fcf_growth'].describe()['75%']
lvl_capex_growth = df_metrics['capex_growth'].describe()['75%']
lvl_rev_growth, lvl_gpf_growth, lvl_fcf_growth, lvl_capex_growth

mask = ((df_metrics['revenue_growth'] >= lvl_rev_growth)
        & (df_metrics['gpf_growth'] >= lvl_gpf_growth)
        & (df_metrics['fcf_growth'] >= lvl_fcf_growth)
        & (df_metrics['capex_growth'] >= lvl_capex_growth)
        & (df_metrics['share_growth'] < 0))

df_metrics.loc[mask].sort_values(by=['revenue_growth', 'gpf_growth', 'fcf_growth', 'capex_growth'], ascending=False)

#.sort_values(by=['revenue_growth', 'gpf_recent_margin', 'fcf_recent_margin', 'capex_recent_growth'], ascending=False).head(10)

,Symbol,sector,industry,market_cap,revenue,revenue_growth,gpf_growth,fcf_growth,capex_growth,fcf_recent_margin,fcf_past_margin,gpf_recent_margin,gpf_past_margin,capex_recent_growth,capex_past_growth,safety_recent,safety_past,share_growth
920,AROC,Energy,Oil & Gas Equipment & Services,4578901504,3.824300e+08,30.90,9.36,79.47,47.35,78.34,43.65,48.50,44.35,35.32,23.97,1229.51,1694.65,-0.34
514,CF,Basic Materials,Agricultural Inputs,12826639360,1.659000e+09,21.09,17.56,31.86,106.11,85.05,64.50,38.10,32.41,20.92,10.15,190.23,207.09,-10.23
156,MLCO,Consumer Cyclical,Resorts & Casinos,3168268800,1.309507e+09,11.43,5.46,389.74,88.14,NaN,1.17,38.83,36.82,NaN,NaN,1954.77,2311.57,-3.78
